In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    SelectKBest,
    f_classif,
    mutual_info_classif
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
np.random.seed(42)

n = 500

df = pd.DataFrame({
    "Customer_ID": range(1001, 1001 + n),
    "Age": np.random.randint(18, 65, n),
    "Annual_Income": np.random.randint(200000, 1500000, n),
    "Purchase_Amount": np.random.randint(500, 20000, n),
    "Quantity": np.random.randint(1, 10, n),
    "Discount": np.random.uniform(0, 0.5, n),
    "Product_Category": np.random.choice(
        ["Electronics", "Clothing", "Furniture", "Beauty"], n
    ),
    "Payment_Method": np.random.choice(
        ["Card", "UPI", "Cash", "Wallet"], n
    ),
    "Customer_Rating": np.random.randint(1, 6, n),
    "Previous_Purchases": np.random.randint(0, 30, n)
})

df["Purchase_Date"] = pd.date_range(
    start="2025-01-01",
    periods=n,
    freq="D"
)

df["Repeat_Purchase"] = (
    (df["Previous_Purchases"] >= 10)
    | (df["Customer_Rating"] >= 4)
).astype(int)

print("Assessment Dataset Created!")
print("Dataset Shape:", df.shape)

display(df.head())

Assessment Dataset Created!
Dataset Shape: (500, 12)


,Customer_ID,Age,Annual_Income,Purchase_Amount,Quantity,Discount,Product_Category,Payment_Method,Customer_Rating,Previous_Purchases,Purchase_Date,Repeat_Purchase
0,1001,56,1252576,4518,4,0.410903,Furniture,Wallet,4,16,2025-01-01,1
1,1002,46,925948,11802,4,0.000618,Furniture,Cash,3,5,2025-01-02,0
2,1003,32,1087474,2470,4,0.150427,Beauty,Cash,1,0,2025-01-03,0
3,1004,60,451492,8889,2,0.490618,Furniture,Wallet,2,4,2025-01-04,0
4,1005,25,691072,18086,4,0.237443,Electronics,Wallet,5,24,2025-01-05,1


In [3]:
# 1. Total purchase value
df["Total_Purchase_Value"] = (
    df["Purchase_Amount"] * df["Quantity"]
)

# 2. Discount amount
df["Discount_Amount"] = (
    df["Purchase_Amount"] * df["Discount"]
)

# 3. Customer purchase history category
df["Purchase_Category"] = pd.cut(
    df["Previous_Purchases"],
    bins=[-1, 5, 15, 30],
    labels=["Low", "Medium", "High"]
)

# 4. Customer age group
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[17, 25, 40, 55, 65],
    labels=["Young", "Adult", "Middle_Aged", "Senior"]
)

# 5. Purchase value per item
df["Value_Per_Item"] = (
    df["Purchase_Amount"] / df["Quantity"]
)

print("Five New Features Created!")
display(df.head())

Five New Features Created!


,Customer_ID,Age,Annual_Income,Purchase_Amount,Quantity,Discount,Product_Category,Payment_Method,Customer_Rating,Previous_Purchases,Purchase_Date,Repeat_Purchase,Total_Purchase_Value,Discount_Amount,Purchase_Category,Age_Group,Value_Per_Item
0,1001,56,1252576,4518,4,0.410903,Furniture,Wallet,4,16,2025-01-01,1,18072,1856.460757,High,Senior,1129.5
1,1002,46,925948,11802,4,0.000618,Furniture,Cash,3,5,2025-01-02,0,47208,7.297647,Low,Middle_Aged,2950.5
2,1003,32,1087474,2470,4,0.150427,Beauty,Cash,1,0,2025-01-03,0,9880,371.553589,Low,Adult,617.5
3,1004,60,451492,8889,2,0.490618,Furniture,Wallet,2,4,2025-01-04,0,17778,4361.105066,Low,Senior,4444.5
4,1005,25,691072,18086,4,0.237443,Electronics,Wallet,5,24,2025-01-05,1,72344,4294.399013,High,Young,4521.5


In [4]:
# Create 100 customer groups across 500 records
df["Customer_Group_ID"] = (
    1001 + (np.arange(len(df)) % 100)
)

# Calculate customer-level statistics
customer_agg = df.groupby("Customer_Group_ID").agg(
    Total_Orders=("Purchase_Amount", "count"),
    Average_Purchase=("Purchase_Amount", "mean"),
    Total_Spending=("Purchase_Amount", "sum"),
    Average_Quantity=("Quantity", "mean"),
    Average_Rating=("Customer_Rating", "mean")
).reset_index()

print("Customer Aggregation Completed!")

display(customer_agg.head(10))

Customer Aggregation Completed!


,Customer_Group_ID,Total_Orders,Average_Purchase,Total_Spending,Average_Quantity,Average_Rating
0,1001,5,11662.8,58314,5.4,3.0
1,1002,5,11012.4,55062,4.6,3.0
2,1003,5,5470.6,27353,4.8,1.8
3,1004,5,10674.6,53373,3.2,2.4
4,1005,5,15680.0,78400,3.8,3.4
5,1006,5,15170.8,75854,4.6,3.4
6,1007,5,7915.6,39578,5.4,1.8
7,1008,5,7917.6,39588,5.6,3.2
8,1009,5,8878.2,44391,6.4,3.2
9,1010,5,7559.4,37797,4.4,2.4


In [5]:
df = df.merge(
    customer_agg,
    on="Customer_Group_ID",
    how="left"
)

print("Customer Aggregation Features Added!")

display(df[[
    "Customer_Group_ID",
    "Total_Orders",
    "Average_Purchase",
    "Total_Spending",
    "Average_Quantity",
    "Average_Rating"
]].head())

Customer Aggregation Features Added!


,Customer_Group_ID,Total_Orders,Average_Purchase,Total_Spending,Average_Quantity,Average_Rating
0,1001,5,11662.8,58314,5.4,3.0
1,1002,5,11012.4,55062,4.6,3.0
2,1003,5,5470.6,27353,4.8,1.8
3,1004,5,10674.6,53373,3.2,2.4
4,1005,5,15680.0,78400,3.8,3.4


In [6]:
df["Purchase_Date"] = pd.to_datetime(df["Purchase_Date"])

df["Purchase_Year"] = df["Purchase_Date"].dt.year
df["Purchase_Month"] = df["Purchase_Date"].dt.month
df["Purchase_Day"] = df["Purchase_Date"].dt.day
df["Purchase_DayOfWeek"] = df["Purchase_Date"].dt.dayofweek

df["Is_Weekend"] = (
    df["Purchase_DayOfWeek"] >= 5
).astype(int)

df["Purchase_Quarter"] = df["Purchase_Date"].dt.quarter

print("Date Features Extracted Successfully!")

display(df[[
    "Purchase_Date",
    "Purchase_Year",
    "Purchase_Month",
    "Purchase_Day",
    "Purchase_DayOfWeek",
    "Is_Weekend",
    "Purchase_Quarter"
]].head())

Date Features Extracted Successfully!


,Purchase_Date,Purchase_Year,Purchase_Month,Purchase_Day,Purchase_DayOfWeek,Is_Weekend,Purchase_Quarter
0,2025-01-01,2025,1,1,2,0,1
1,2025-01-02,2025,1,2,3,0,1
2,2025-01-03,2025,1,3,4,0,1
3,2025-01-04,2025,1,4,5,1,1
4,2025-01-05,2025,1,5,6,1,1


In [7]:
# Purchase amount and quantity
df["Amount_Quantity_Interaction"] = (
    df["Purchase_Amount"] * df["Quantity"]
)

# Income and purchase amount
df["Income_Purchase_Ratio"] = (
    df["Purchase_Amount"] / df["Annual_Income"]
)

# Discount and quantity
df["Discount_Quantity_Interaction"] = (
    df["Discount"] * df["Quantity"]
)

# Age and purchase amount
df["Age_Purchase_Interaction"] = (
    df["Age"] * df["Purchase_Amount"]
)

print("Interaction Features Created Successfully!")

display(df[[
    "Purchase_Amount",
    "Quantity",
    "Amount_Quantity_Interaction",
    "Income_Purchase_Ratio",
    "Discount_Quantity_Interaction",
    "Age_Purchase_Interaction"
]].head())

Interaction Features Created Successfully!


,Purchase_Amount,Quantity,Amount_Quantity_Interaction,Income_Purchase_Ratio,Discount_Quantity_Interaction,Age_Purchase_Interaction
0,4518,4,18072,0.003607,1.643613,253008
1,11802,4,47208,0.012746,0.002473,542892
2,2470,4,9880,0.002271,0.601706,79040
3,8889,2,17778,0.019688,0.981236,533340
4,18086,4,72344,0.026171,0.949773,452150


In [8]:
category_frequency = (
    df["Product_Category"].value_counts(normalize=True)
)

df["Product_Category_Frequency"] = (
    df["Product_Category"].map(category_frequency)
)

payment_frequency = (
    df["Payment_Method"].value_counts(normalize=True)
)

df["Payment_Method_Frequency"] = (
    df["Payment_Method"].map(payment_frequency)
)

print("Frequency Encoding Completed!")

display(df[[
    "Product_Category",
    "Product_Category_Frequency",
    "Payment_Method",
    "Payment_Method_Frequency"
]].head(10))

Frequency Encoding Completed!


,Product_Category,Product_Category_Frequency,Payment_Method,Payment_Method_Frequency
0,Furniture,0.250,Wallet,0.248
1,Furniture,0.250,Cash,0.272
2,Beauty,0.242,Cash,0.272
3,Furniture,0.250,Wallet,0.248
4,Electronics,0.278,Wallet,0.248
5,Electronics,0.278,Cash,0.272
6,Furniture,0.250,Card,0.250
7,Electronics,0.278,Cash,0.272
8,Beauty,0.242,Cash,0.272
9,Electronics,0.278,Wallet,0.248


In [9]:
from sklearn.model_selection import train_test_split

# Split the data first
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Repeat_Purchase"]
)

# Calculate target encoding from training data only
target_means = train_df.groupby(
    "Product_Category"
)["Repeat_Purchase"].mean()

global_mean = train_df["Repeat_Purchase"].mean()

# Apply mappings to training and testing data
train_df["Product_Category_Target_Encoded"] = (
    train_df["Product_Category"].map(target_means).fillna(global_mean)
)

test_df["Product_Category_Target_Encoded"] = (
    test_df["Product_Category"].map(target_means).fillna(global_mean)
)

print("Target Encoding Completed!")

display(train_df[[
    "Product_Category",
    "Repeat_Purchase",
    "Product_Category_Target_Encoded"
]].head(10))

Target Encoding Completed!


,Product_Category,Repeat_Purchase,Product_Category_Target_Encoded
212,Clothing,1,0.731183
384,Electronics,1,0.823009
301,Electronics,1,0.823009
497,Beauty,1,0.801980
139,Furniture,1,0.795699
472,Electronics,0,0.823009
486,Furniture,1,0.795699
467,Electronics,1,0.823009
277,Beauty,1,0.801980
410,Beauty,0,0.801980


In [10]:
categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns

cardinality = pd.DataFrame({
    "Feature": categorical_columns,
    "Unique_Values": [
        df[col].nunique() for col in categorical_columns
    ]
})

cardinality = cardinality.sort_values(
    by="Unique_Values",
    ascending=False
)

display(cardinality)

,Feature,Unique_Values
0,Product_Category,4
1,Payment_Method,4
3,Age_Group,4
2,Purchase_Category,3


In [11]:
df["Log_Purchase_Amount"] = np.log1p(
    df["Purchase_Amount"]
)

df["Log_Annual_Income"] = np.log1p(
    df["Annual_Income"]
)

print("Log Transformation Completed!")

display(df[[
    "Purchase_Amount",
    "Log_Purchase_Amount",
    "Annual_Income",
    "Log_Annual_Income"
]].head())

Log Transformation Completed!


,Purchase_Amount,Log_Purchase_Amount,Annual_Income,Log_Annual_Income
0,4518,8.416046,1252576,14.040714
1,11802,9.376109,925948,13.738574
2,2470,7.812378,1087474,13.899369
3,8889,9.092682,451492,13.020315
4,18086,9.802949,691072,13.446001


In [12]:
from sklearn.feature_selection import (
    SelectKBest,
    f_classif,
    mutual_info_classif
)
from sklearn.ensemble import RandomForestClassifier

# Exclude target and identifier columns
exclude_columns = [
    "Repeat_Purchase",
    "Customer_ID",
    "Customer_Group_ID"
]

X = df.select_dtypes(include=np.number).drop(
    columns=exclude_columns,
    errors="ignore"
)

y = df["Repeat_Purchase"]

# Replace infinite values and missing values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# Split before fitting selection methods
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (400, 29)
Testing Shape: (100, 29)


In [13]:
k = min(5, X_train.shape[1])

anova_selector = SelectKBest(
    score_func=f_classif,
    k=k
)

anova_selector.fit(X_train, y_train)

anova_features = X_train.columns[
    anova_selector.get_support()
]

print("ANOVA Selected Features:")
print(anova_features.tolist())

ANOVA Selected Features:
['Customer_Rating', 'Previous_Purchases', 'Total_Purchase_Value', 'Average_Rating', 'Amount_Quantity_Interaction']


In [14]:
k = min(5, X_train.shape[1])

anova_selector = SelectKBest(
    score_func=f_classif,
    k=k
)

anova_selector.fit(X_train, y_train)

anova_features = X_train.columns[
    anova_selector.get_support()
]

print("ANOVA Selected Features:")
print(anova_features.tolist())

ANOVA Selected Features:
['Customer_Rating', 'Previous_Purchases', 'Total_Purchase_Value', 'Average_Rating', 'Amount_Quantity_Interaction']


In [15]:
mi_scores = mutual_info_classif(
    X_train,
    y_train,
    random_state=42
)

mi_results = pd.DataFrame({
    "Feature": X_train.columns,
    "Mutual_Information": mi_scores
}).sort_values(
    by="Mutual_Information",
    ascending=False
)

print("Top Mutual Information Features:")
display(mi_results.head(10))

Top Mutual Information Features:


,Feature,Mutual_Information
6,Previous_Purchases,0.286674
5,Customer_Rating,0.118073
27,Log_Purchase_Amount,0.051671
2,Purchase_Amount,0.050835
26,Payment_Method_Frequency,0.031593
0,Age,0.022829
3,Quantity,0.020270
25,Product_Category_Frequency,0.017472
7,Total_Purchase_Value,0.017407
21,Amount_Quantity_Interaction,0.017407


In [16]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print("Top Random Forest Features:")
display(rf_importance.head(10))

Top Random Forest Features:


,Feature,Importance
6,Previous_Purchases,0.439430
5,Customer_Rating,0.185499
14,Average_Rating,0.025008
23,Discount_Quantity_Interaction,0.023001
21,Amount_Quantity_Interaction,0.022248
2,Purchase_Amount,0.022170
4,Discount,0.021971
1,Annual_Income,0.021424
27,Log_Purchase_Amount,0.020010
28,Log_Annual_Income,0.019121
